### Data
We use the course data in the folder `Data` (place it next to this notebook):
* `GPL1261-56135.tsv` – annotation of the Affymetrix Mouse Genome 430 2.0 array (45,101 probes, 64 MB on disk).
* `primates_diet.tsv` – expression of the 43,014 probes in 24 mouse samples: four diets (Chimpanzee, FastFood, HumanCafe, Pellet) × 2 batches × 3 replicates.

These files are small enough to experiment with in class, but large enough to *see* the effects of the
techniques. Everything we do scales unchanged to files that are many times larger than your RAM.

In [1]:
import csv
import heapq
import sqlite3
import time
import tracemalloc
from collections import Counter
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

# --- locate the course data folder -------------------------------------------
CANDIDATES = [Path("Data"), Path("../Data"), Path("../../Data")]
DATA_DIR = next((p for p in CANDIDATES if (p / "primates_diet.tsv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Put the course folder 'Data' (with GPL1261-56135.tsv and "
                            "primates_diet.tsv) next to this notebook.")

ANNOTATION = DATA_DIR / "GPL1261-56135.tsv"
EXPRESSION = DATA_DIR / "primates_diet.tsv"
LONG_FILE = Path("primates_diet_long.tsv")    # created in Part 2
DB_FILE = Path("primates_diet.sqlite")         # created in Part 3


@contextmanager
def measure(label):
    # Print wall-clock time and peak Python memory of the indented block.
    tracemalloc.start()
    start = time.perf_counter()
    try:
        yield
    finally:
        elapsed = time.perf_counter() - start
        _, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        print(f"{label:<42} {elapsed:6.2f} s   peak memory {peak / 1e6:7.1f} MB")


def mb(df):
    # Memory of a DataFrame/Series in MB, including the content of strings.
    return df.memory_usage(deep=True).sum() / 1e6 if isinstance(df, pd.DataFrame) \
        else df.memory_usage(deep=True) / 1e6


for f in (ANNOTATION, EXPRESSION):
    print(f"{f.name:<25} {f.stat().st_size / 1e6:6.1f} MB on disk")

GPL1261-56135.tsv           64.0 MB on disk
primates_diet.tsv            3.2 MB on disk


<details><summary><b>Automatic checks for the exercises</b> – run the next cell once, you do not need to read it.</summary>
The functions <code>check_exercise_1()</code>, <code>check_exercise_2()</code> and <code>check_exercise_3()</code> compare
your results with a reference computed in memory with pandas.
</details>

In [2]:
def _report(task, func):
    # Run one check; print a tick, a cross or 'not solved yet'.
    try:
        ok, detail = func()
        print(("✔ " if ok else "✘ ") + task + (f" – {detail}" if detail else ""))
    except NotImplementedError:
        print(f"… {task} – not solved yet")
    except Exception as err:                                   # noqa: BLE001
        print(f"✘ {task} – {type(err).__name__}: {err}")


def _reference_expression():
    expr = pd.read_csv(EXPRESSION, sep="\t")
    values = expr.set_index("ID_REF")
    chimp = values.filter(like="Chimpanzee").mean(axis=1)
    pellet = values.filter(like="Pellet").mean(axis=1)
    return expr, values, (chimp / pellet).sort_values(ascending=False)


def check_exercise_1(stream_expression, sample_means, count_high_probes, top_chimp_vs_pellet):
    expr, values, ratio = _reference_expression()

    def a():
        items = stream_expression(EXPRESSION)
        first = next(iter(items))
        expected = ("1415670_at", [57, 73, 83, 33, 24, 37, 60, 65, 59, 40, 24, 39,
                                   49, 61, 73, 39, 38, 23, 97, 72, 88, 58, 62, 71])
        n = 1 + sum(1 for _ in items)
        ok = first == expected and n == len(expr) and not isinstance(items, list)
        return ok, f"{n} records, first = {first[0]}" + ("" if ok else " (expected a lazy generator of 43014 tuples)")

    def b():
        means = sample_means(EXPRESSION)
        ok = all(abs(means[s] - values[s].mean()) < 1e-6 for s in values.columns)
        return ok, f"mean of {values.columns[0]} = {means[values.columns[0]]:.2f}"

    def c():
        n = count_high_probes(EXPRESSION, threshold=1000)
        expected = int((values.max(axis=1) > 1000).sum())
        return n == expected, f"{n} probes (expected {expected})"

    def d():
        top = top_chimp_vs_pellet(EXPRESSION, k=5)
        ok = [p for _, p in top] == list(ratio.index[:5])
        return ok, ", ".join(f"{p}: {r:.2f}" for r, p in top)

    for name, f in [("1A stream_expression", a), ("1B sample_means", b),
                    ("1C count_high_probes", c), ("1D top_chimp_vs_pellet (bonus)", d)]:
        _report(name, f)


def _reference_annotation():
    ann = pd.read_csv(ANNOTATION, sep="\t", usecols=["ID", "Gene Symbol", "Sequence Type"])
    ann = ann[(ann["Sequence Type"] != "Control sequence") & ann["Gene Symbol"].notna()]
    return pd.DataFrame({"probe_id": ann["ID"].to_numpy(),
                         "symbol": ann["Gene Symbol"].str.split(" /// ").str[0].to_numpy()})


def _reference_diet_means():
    expr = pd.read_csv(EXPRESSION, sep="\t").melt(id_vars="ID_REF", var_name="sample", value_name="value")
    expr["diet"] = expr["sample"].str.extract(r"^(\w+?)Diet", expand=False)
    merged = expr.merge(_reference_annotation(), left_on="ID_REF", right_on="probe_id")
    return merged.groupby(["symbol", "diet"])["value"].mean().unstack("diet")


def check_exercise_2(load_expression_small, load_annotation_small, diet_means_chunked, top_fastfood_vs_pellet):
    default_mb = mb(pd.read_csv(EXPRESSION, sep="\t"))
    state = {}

    def a():
        small = load_expression_small(EXPRESSION)
        ratio = default_mb / mb(small)
        ok = ratio >= 2 and int(small.select_dtypes("number").to_numpy(dtype="int64").sum()) == \
            int(pd.read_csv(EXPRESSION, sep="\t").iloc[:, 1:].to_numpy().sum())
        return ok, f"{default_mb:.1f} MB -> {mb(small):.1f} MB ({ratio:.1f}x smaller, goal: at least 2x)"

    def b():
        ann = load_annotation_small(ANNOTATION)
        ref = _reference_annotation()
        ok = list(ann.columns) == ["probe_id", "symbol"] and len(ann) == len(ref) \
            and isinstance(ann["symbol"].dtype, pd.CategoricalDtype)
        state["annotation"] = ann
        return ok, f"{len(ann)} probes, {mb(ann):.1f} MB, symbol dtype = {ann['symbol'].dtype}"

    def c():
        ann = state.get("annotation", _reference_annotation())
        means = diet_means_chunked(LONG_FILE, ann, chunksize=100_000)
        ref = _reference_diet_means()
        means = means.loc[ref.index, ref.columns]
        state["means"] = means
        return bool(np.allclose(means.to_numpy(), ref.to_numpy())), f"{means.shape[0]} genes x {means.shape[1]} diets"

    def d():
        means = state.get("means", _reference_diet_means())
        top = top_fastfood_vs_pellet(means, k=10)
        ref = np.log2((means["FastFood"] + 1) / (means["Pellet"] + 1)).sort_values(ascending=False).head(10)
        ok = list(top.index) == list(ref.index) and np.allclose(top.to_numpy(), ref.to_numpy())
        return ok, ", ".join(f"{g} ({v:.2f})" for g, v in top.head(3).items()) + ", ..."

    for name, f in [("2A load_expression_small", a), ("2B load_annotation_small", b),
                    ("2C diet_means_chunked", c), ("2D top_fastfood_vs_pellet", d)]:
        _report(name, f)


def check_exercise_3(query):
    def a():
        if query is None:
            raise NotImplementedError
        with sqlite3.connect(DB_FILE) as con:
            result = pd.read_sql_query(query, con)
            ref = pd.read_sql_query(
                "SELECT a.symbol, AVG(e.value) AS mean_value FROM expression e "
                "JOIN annotation a ON e.probe_id = a.probe_id WHERE e.diet = 'HumanCafe' "
                "GROUP BY a.symbol ORDER BY mean_value DESC LIMIT 5", con)
        ok = list(result.iloc[:, 0]) == list(ref["symbol"]) and np.allclose(result.iloc[:, 1], ref["mean_value"])
        return ok, ", ".join(result.iloc[:, 0].astype(str))

    _report("3A SQL query", a)

---
# Streaming in plain Python

## 1.1 Reading everything vs. streaming
A file object is an **iterator over lines**. `f.readlines()` builds a list with *all* lines in memory;
`for line in f` keeps only *one* line at a time. Both visit every line once (O(n) time),
but the memory need is O(n) vs. O(1).

In [3]:
with measure("readlines(): whole file as a list"):
    with open(ANNOTATION, encoding="utf-8") as f:
        lines = f.readlines()
    n_lines = len(lines)
print("lines:", n_lines)
del lines


with measure("for line in f: one line at a time"):
    n_lines = 0
    with open(ANNOTATION, encoding="utf-8") as f:
        for line in f:
            n_lines += 1
print("lines:", n_lines)

readlines(): whole file as a list            0.85 s   peak memory    66.2 MB
lines: 45102
for line in f: one line at a time            0.65 s   peak memory     0.1 MB
lines: 45102


## 1.2 Generators: functions that *yield*
A function containing `yield` returns a **generator**. Its body runs only when someone asks for the next
value, and it pauses after each `yield`. You already know this from `LinkedList.__iter__` in Session 8.

We use the `csv` module because fields of the annotation file can contain quotes and separators.

In [6]:
def read_records(path):
    with open(path, encoding="utf-8", newline="") as f:
        reader = csv.reader(f, delimiter="\t")
        header = next(reader)
        for fields in reader:
            yield dict(zip(header, fields))


records = read_records(ANNOTATION)
print(type(records))
first = next(records)
print(first["ID"], first["Gene Symbol"], first["Species Scientific Name"])

second = next(records)
print(second["ID"], second["Gene Symbol"], second["Species Scientific Name"])

<class 'generator'>
1415670_at Copg1 Mus musculus
1415671_at Atp6v0d1 Mus musculus


## 1.3 Generator pipelines
Small generators can be chained like a pipeline: *read → filter → transform → aggregate*.
Each record flows through the whole pipeline before the next one is read.

In [8]:
def only_mouse(records):
    for r in records:
        if r["Species Scientific Name"] == "Mus musculus":
            yield r

def first_symbol(records):
    for r in records:
        if r["Gene Symbol"]:
            yield r["Gene Symbol"].split(" /// ")[0]


pipeline = first_symbol(only_mouse(read_records(ANNOTATION)))
with measure("pipeline: Counter over a generator pipeline"):
    counts = Counter(pipeline)

print(len(counts), "different gene symbols")
print(counts.most_common(5))


n_controls = sum(1 for r in read_records(ANNOTATION) if r["Sequence Type"] == "Control sequence")
print("control probes:", n_controls)

pipeline: Counter over a generator pipeline   3.67 s   peak memory     1.6 MB
21495 different gene symbols
[('Igh-VJ558', 16), ('Rbm39', 12), ('abParts', 12), ('Calm1', 11), ('Kcnq2', 11)]
control probes: 64


## 1.4 Streaming statistics
Many statistics can be **updated record by record**: count, sum, mean, minimum/maximum, variance
(via the sum of squares) and even the *k* largest values (`heapq.nlargest` keeps only *k* items).
The median cannot – it needs all values (or a sorted file).

In [10]:
def column_values(path, column):
    with open(path, encoding="utf-8") as f:
        header = f.readline().rstrip("\n").split("\t")
        idx = header.index(column)
        for line in f:
            yield int (line.rstrip("\n").split("\t")[idx])

count = total = total_sq = 0
maximum = float("-inf")
for v in column_values(EXPRESSION, "PelletDiet1.1"):
    count += 1
    total += v
    total_sq += v*v
    maximum = max(maximum, v)


mean = total / count
sd = ((total_sq - count * mean ** 2) / (count - 1)) ** 0.5
print(f"n = {count}, mean = {mean:.1f}, sd = {sd:.1f}, maximum = {maximum}")


n = 43014, mean = 68.1, sd = 283.9, maximum = 7575


---
# Exercise 1 – Streaming the expression file
Work **only with streaming**: never build a list of all lines or records.
The helper `read_header` is given.

* **1A** Write the generator `stream_expression(path)` that yields one tuple `(probe_id, values)` per data line
  of `primates_diet.tsv`, where `values` is a list of 24 integers.
* **1B** Write `sample_means(path)` that returns a dictionary *sample name → mean expression* using
  running sums over `stream_expression`.
* **1C** Write `count_high_probes(path, threshold)` that counts the probes whose maximum over all samples is
  larger than `threshold`.
* **1D (bonus)** Write `top_chimp_vs_pellet(path, k)` that returns the *k* probes with the largest ratio
  *mean(Chimpanzee samples) / mean(Pellet samples)* as a list of `(ratio, probe_id)`, largest first.
  Use `heapq.nlargest` on a generator expression.
* **1E (discussion)** Use `measure(...)` to compare the peak memory of your `sample_means` with a version that
  first calls `f.readlines()`. What changes if the file were 100× larger?

In [11]:
def read_header(path):
    # Return the list of column names of a tab-separated file.
    with open(path, encoding="utf-8") as f:
        return f.readline().rstrip("\n").split("\t")


print(read_header(EXPRESSION)[:4], "...")

['ID_REF', 'ChimpanzeeDiet1.1', 'ChimpanzeeDiet1.2', 'ChimpanzeeDiet1.3'] ...


In [12]:
def stream_expression(path):
    with open(path, encoding="utf-8") as f:
        next(f)                                       # skip the header
        for line in f:
            fields = line.rstrip("\n").split("\t")
            yield fields[0], [int(x) for x in fields[1:]]


def sample_means(path):
    samples = read_header(path)[1:]
    sums = [0] * len(samples)
    n = 0
    for _, values in stream_expression(path):
        n += 1
        for i, v in enumerate(values):
            sums[i] += v
    return {s: total / n for s, total in zip(samples, sums)}


def count_high_probes(path, threshold=1000):
    return sum(1 for _, values in stream_expression(path) if max(values) > threshold)


def top_chimp_vs_pellet(path, k=5):
    samples = read_header(path)[1:]
    chimp = [i for i, s in enumerate(samples) if s.startswith("Chimpanzee")]
    pellet = [i for i, s in enumerate(samples) if s.startswith("Pellet")]
    ratios = (
        (sum(values[i] for i in chimp) / sum(values[i] for i in pellet), probe)
        for probe, values in stream_expression(path)
    )
    return heapq.nlargest(k, ratios)

In [13]:
check_exercise_1(stream_expression, sample_means, count_high_probes, top_chimp_vs_pellet)

✔ 1A stream_expression – 43014 records, first = 1415670_at
✔ 1B sample_means – mean of ChimpanzeeDiet1.1 = 61.76
✔ 1C count_high_probes – 739 probes (expected 739)
✔ 1D top_chimp_vs_pellet (bonus) – 1423257_at: 9.65, 1418486_at: 8.17, 1425303_at: 7.41, 1417017_at: 7.13, 1417991_at: 5.82


In [14]:
def sample_means_readlines(path):
    with open(path, encoding="utf-8") as f:
        lines = f.readlines()                         # whole file in memory
    samples = lines[0].rstrip("\n").split("\t")[1:]
    rows = [[int(x) for x in line.rstrip("\n").split("\t")[1:]] for line in lines[1:]]
    return {s: sum(r[i] for r in rows) / len(rows) for i, s in enumerate(samples)}


with measure("streaming sample_means"):
    sample_means(EXPRESSION)
with measure("readlines() + list of rows"):
    sample_means_readlines(EXPRESSION)

streaming sample_means                       2.21 s   peak memory     0.0 MB
readlines() + list of rows                   2.06 s   peak memory    17.9 MB


# pandas for large data

## 2.1 Load only what you need: `usecols`, `nrows`, `dtype`
By default pandas stores text as Python objects (`object`) and numbers as 64-bit types.
`memory_usage(deep=True)` shows the real memory, including the content of strings.

In [15]:
with measure("read_csv: all 16 columns"):
    ann_full = pd.read_csv(ANNOTATION, sep="\t", low_memory=False)
print(f"all columns:      {mb(ann_full):6.1f} MB")
print((ann_full.memory_usage(deep=True) / 1e6).sort_values(ascending=False).head(4).round(1))

peek = pd.read_csv(ANNOTATION, sep="\t", nrows=5)          # look at the first rows only
print(peek.columns.tolist()[:6], "...")

with measure("read_csv: 3 columns, category dtype"):
    ann_small = pd.read_csv(ANNOTATION, sep="\t",
                            usecols=["ID", "Gene Symbol", "Sequence Type"],
                            dtype={"Sequence Type": "category"})
print(f"3 columns:        {mb(ann_small):6.1f} MB")
del ann_full

read_csv: all 16 columns                     1.18 s   peak memory    56.0 MB
all columns:        96.7 MB
Gene Ontology Biological Process    22.7
Gene Ontology Molecular Function    16.4
Target Description                  12.6
Gene Ontology Cellular Component    11.7
dtype: float64
['ID', 'GB_ACC', 'SPOT_ID', 'Species Scientific Name', 'Annotation Date', 'Sequence Type'] ...
read_csv: 3 columns, category dtype          0.68 s   peak memory     6.7 MB
3 columns:           5.1 MB


## 2.2 Choose small dtypes: downcasting and `category`
* Integers from 4 to 8255 fit into `int16` (–32768 … 32767): 2 bytes instead of 8.
* Columns with few distinct text values (diet, species, chromosome) become `category`:
  every value is stored once, the rows only keep a small integer code.

In [16]:
expr = pd.read_csv(EXPRESSION, sep="\t")
print(f"default dtypes:   {mb(expr):5.1f} MB   {expr.dtypes.value_counts().to_dict()}")

values = expr.iloc[:, 1:]
print("value range:", values.min().min(), "to", values.max().max())

expr_small = expr.copy()
expr_small.iloc[:, 1:] = expr_small.iloc[:, 1:].apply(pd.to_numeric, downcast="integer")
expr_small = expr_small.astype({c: "int16" for c in expr.columns[1:]})
print(f"int16 values:     {mb(expr_small):5.1f} MB")

species = pd.read_csv(ANNOTATION, sep="\t", usecols=["Species Scientific Name"]).iloc[:, 0]
print(f"species object:   {mb(species):5.2f} MB   category: {mb(species.astype('category')):5.2f} MB")

default dtypes:    10.8 MB   {dtype('int64'): 24, dtype('O'): 1}
value range: 4 to 8255
int16 values:       4.6 MB
species object:    2.75 MB   category:  0.05 MB


## 2.3 Writing in chunks: from wide to long format
Large omics tables are often stored in **long format** (one measurement per row). We create such a file
from the expression table – chunk by chunk, so that only a few thousand probes are in memory at a time.
`mode="a"` appends to the file; only the first chunk writes the header.

In [17]:
def to_long(chunk):
    # Wide chunk (probe x 24 samples) -> long rows: probe_id, diet, batch, replicate, value.
    long = chunk.melt(id_vars="ID_REF", var_name="sample", value_name="value")
    # parse the 24 sample names once (not 1 million times): 'FastFoodDiet2.3' -> FastFood, 2, 3
    samples = pd.Series(chunk.columns[1:], index=chunk.columns[1:])
    info = samples.str.extract(r"^(?P<diet>\w+?)Diet(?P<batch>\d)\.(?P<replicate>\d)$")
    long = long.join(info, on="sample").rename(columns={"ID_REF": "probe_id"})
    return long[["probe_id", "diet", "batch", "replicate", "value"]]


LONG_FILE.unlink(missing_ok=True)
with measure("write long file in chunks of 5000 probes"):
    for i, chunk in enumerate(pd.read_csv(EXPRESSION, sep="\t", chunksize=5_000)):
        to_long(chunk).to_csv(LONG_FILE, sep="\t", index=False, mode="a", header=(i == 0))

print(f"{LONG_FILE}: {LONG_FILE.stat().st_size / 1e6:.1f} MB")
print(pd.read_csv(LONG_FILE, sep="\t", nrows=3))

write long file in chunks of 5000 probes    13.41 s   peak memory    26.3 MB
primates_diet_long.tsv: 29.2 MB
     probe_id        diet  batch  replicate  value
0  1415670_at  Chimpanzee      1          1     57
1  1415671_at  Chimpanzee      1          1    283
2  1415672_at  Chimpanzee      1          1    233


## 2.4 Reading in chunks: `chunksize`
With `chunksize`, `read_csv` returns an **iterator of DataFrames** – pandas' version of streaming.
Aggregate every chunk, keep only the small partial results, and combine them at the end.

⚠️ **Combine sums and counts, not means.** The mean of chunk means is wrong when the groups have
different sizes in different chunks.

In [18]:
partials = []
chunk_means = []
with measure("chunked groupby (chunks of 200,000 rows)"):
    for chunk in pd.read_csv(LONG_FILE, sep="\t", usecols=["diet", "value"], chunksize=200_000):
        partials.append(chunk.groupby("diet")["value"].agg(["sum", "count"]))
        chunk_means.append(chunk.groupby("diet")["value"].mean())

combined = pd.concat(partials).groupby(level=0).sum()
combined["mean"] = combined["sum"] / combined["count"]
print(combined)

wrong = pd.concat(chunk_means).groupby(level=0).mean()
print("\nmean of chunk means (wrong):")
print(pd.DataFrame({"correct": combined["mean"], "mean of means": wrong}).round(3))

chunked groupby (chunks of 200,000 rows)     0.53 s   peak memory    15.6 MB
                 sum   count       mean
diet                                   
Chimpanzee  14996001  258084  58.105117
FastFood    15668076  258084  60.709211
HumanCafe   15684709  258084  60.773659
Pellet      15580386  258084  60.369438

mean of chunk means (wrong):
            correct  mean of means
diet                              
Chimpanzee   58.105         55.642
FastFood     60.709         58.844
HumanCafe    60.774         58.739
Pellet       60.369         59.990


# Exercise 2 – An out-of-core pandas pipeline
**Question:** which genes react most strongly to the fast-food diet compared to the pellet (standard) diet?
Pretend that the long file `primates_diet_long.tsv` is too large to load at once.

* **2A** `load_expression_small(path)` – load `primates_diet.tsv` so that it needs **at least 2× less memory**
  than with default dtypes (all values must stay the same).
* **2B** `load_annotation_small(path)` – load only what is needed from the annotation: a DataFrame with the columns
  `probe_id` and `symbol` (in this order). Drop control probes (`Sequence Type == "Control sequence"`) and probes
  without gene symbol; if several symbols are given (`"A /// B"`), keep the first. `symbol` must be `category`.
* **2C** `diet_means_chunked(long_path, annotation, chunksize)` – read the long file in chunks, merge every chunk with
  the annotation, and keep **sums and counts** per `(symbol, diet)`. Return the mean expression as a DataFrame
  with one row per gene symbol and one column per diet.
* **2D** `top_fastfood_vs_pellet(means, k)` – return the *k* genes with the largest
  log2 fold change `log2((FastFood + 1) / (Pellet + 1))` as a Series sorted in descending order.
* **Bonus** Time `diet_means_chunked` with chunk sizes 10,000, 100,000 and 1,000,000. What do you observe and why?

In [19]:
def load_expression_small(path):
    columns = read_header(path)
    return pd.read_csv(path, sep="\t", dtype={c: "int16" for c in columns[1:]})


def load_annotation_small(path):
    ann = pd.read_csv(path, sep="\t", usecols=["ID", "Gene Symbol", "Sequence Type"],
                      dtype={"Sequence Type": "category"})
    ann = ann[(ann["Sequence Type"] != "Control sequence") & ann["Gene Symbol"].notna()]
    return pd.DataFrame({
        "probe_id": ann["ID"].to_numpy(),
        "symbol": ann["Gene Symbol"].str.split(" /// ").str[0].astype("category").to_numpy(),
    }).astype({"symbol": "category"})


def diet_means_chunked(long_path, annotation, chunksize=100_000):
    partials = []
    for chunk in pd.read_csv(long_path, sep="\t", usecols=["probe_id", "diet", "value"],
                             dtype={"diet": "category", "value": "int16"}, chunksize=chunksize):
        merged = chunk.merge(annotation, on="probe_id")          # inner join drops unannotated probes
        partials.append(merged.groupby(["symbol", "diet"], observed=True)["value"].agg(["sum", "count"]))
    totals = pd.concat(partials).groupby(level=["symbol", "diet"], observed=True).sum()
    return (totals["sum"] / totals["count"]).unstack("diet")


def top_fastfood_vs_pellet(means, k=10):
    lfc = np.log2((means["FastFood"] + 1) / (means["Pellet"] + 1))
    return lfc.sort_values(ascending=False).head(k)

In [20]:
check_exercise_2(load_expression_small, load_annotation_small, diet_means_chunked, top_fastfood_vs_pellet)

✔ 2A load_expression_small – 10.8 MB -> 4.6 MB (2.3x smaller, goal: at least 2x)
✔ 2B load_annotation_small – 39653 probes, 4.2 MB, symbol dtype = category
✔ 2C diet_means_chunked – 20943 genes x 4 diets
✔ 2D top_fastfood_vs_pellet – Cyp4a14 (6.13), Cyp4a10 (3.35), Dmbt1 (3.14), ...


In [21]:
annotation = load_annotation_small(ANNOTATION)
for size in (10_000, 100_000, 1_000_000):
    with measure(f"chunksize = {size:,}"):
        diet_means_chunked(LONG_FILE, annotation, chunksize=size)

chunksize = 10,000                          10.84 s   peak memory    42.1 MB
chunksize = 100,000                          1.66 s   peak memory    21.8 MB
chunksize = 1,000,000                        0.92 s   peak memory    89.6 MB


**Bonus – discussion.** Very small chunks are slow: every chunk pays a fixed overhead (parsing setup, merge,
groupby, Python loop). Very large chunks are fast but need more memory – with 1,000,000 rows the whole file
is one chunk. A chunk size of roughly 10⁵–10⁶ rows is a good compromise: memory stays bounded and the overhead
per row is small. Choose the chunk size by the memory you can afford, not by the file size.

# Beyond memory

## 3.1 A database on disk: SQLite
`sqlite3` is part of the Python standard library. A SQLite database is a single file; queries run **on disk**
and only the result comes back into memory. pandas writes to it with `to_sql` and reads with `read_sql_query`.

In [22]:
DB_FILE.unlink(missing_ok=True)
annotation_db = pd.read_csv(ANNOTATION, sep="\t", usecols=["ID", "Gene Symbol", "Sequence Type"])
annotation_db = annotation_db[(annotation_db["Sequence Type"] != "Control sequence")
                              & annotation_db["Gene Symbol"].notna()]
annotation_db = pd.DataFrame({"probe_id": annotation_db["ID"],
                              "symbol": annotation_db["Gene Symbol"].str.split(" /// ").str[0]})

with sqlite3.connect(DB_FILE) as con:
    with measure("load long file into SQLite in chunks"):
        for chunk in pd.read_csv(LONG_FILE, sep="\t", chunksize=200_000):
            chunk.to_sql("expression", con, if_exists="append", index=False)
    annotation_db.to_sql("annotation", con, index=False)
    con.execute("CREATE INDEX idx_expression_probe ON expression(probe_id)")
    con.execute("CREATE INDEX idx_annotation_probe ON annotation(probe_id)")

    query = "SELECT diet, COUNT(*) AS n, AVG(value) AS mean_value FROM expression GROUP BY diet"
    print(pd.read_sql_query(query, con))

print(f"{DB_FILE}: {DB_FILE.stat().st_size / 1e6:.1f} MB")

load long file into SQLite in chunks         3.09 s   peak memory    35.8 MB
         diet       n  mean_value
0  Chimpanzee  258084   58.105117
1    FastFood  258084   60.709211
2   HumanCafe  258084   60.773659
3      Pellet  258084   60.369438
primates_diet.sqlite: 56.1 MB


## 3.2 Choosing a strategy
| Situation | Strategy |
|---|---|
| Data fits comfortably into RAM | pandas with `usecols`, small dtypes, `category` |
| Slightly larger than RAM, one pass is enough | stream with generators, or `read_csv(chunksize=...)` + combine partial results |
| Analysed many times | convert once to a **columnar binary format** (Parquet/Feather via `pyarrow`) or a database – loading is much faster and you read only the needed columns |
| Much larger than RAM, complex queries | out-of-core engines: **SQLite/DuckDB** (SQL on files), **Polars** (lazy & streaming), **Dask** (pandas-like, parallel) |

**Complexity view (Session 9):** streaming needs O(n) time but only O(1) – or O(k) for top-*k* – memory.
Operations such as sorting or the exact median need all values: they require O(n) memory or an *external*
(on-disk) algorithm.

*Parquet, DuckDB, Polars and Dask need extra packages (`pip install pyarrow duckdb polars dask`);
they are not needed for this notebook.*

# Exercise 3 – SQL and choosing a strategy
* **3A** Write a SQL query (string `QUERY_TOP5`) on `primates_diet.sqlite` that returns the **5 gene symbols with the
  highest mean expression in the HumanCafe diet**: two columns (symbol, mean value), highest first.
  *Hint:* `JOIN annotation a ON e.probe_id = a.probe_id`, `WHERE`, `GROUP BY`, `ORDER BY ... DESC`, `LIMIT`.
* **3B** For each scenario, decide on a strategy and justify it in one sentence (write your answers in the markdown cell):
  1. Count the reads in a 20 GB gzip-compressed FASTQ file (4 lines per read) on a laptop with 16 GB RAM.
  2. A 300 MB CSV file with expression values that your group analyses interactively every day.
  3. The mean expression per gene and condition in a 50 GB long-format table.
  4. The **median** expression per gene in the same 50 GB table.

In [23]:
QUERY_TOP5 = '''
SELECT a.symbol, AVG(e.value) AS mean_value
FROM expression AS e
JOIN annotation AS a ON e.probe_id = a.probe_id
WHERE e.diet = 'HumanCafe'
GROUP BY a.symbol
ORDER BY mean_value DESC
LIMIT 5
'''

In [24]:
check_exercise_3(QUERY_TOP5)
if QUERY_TOP5 is not None:
    with sqlite3.connect(DB_FILE) as con:
        print(pd.read_sql_query(QUERY_TOP5, con))

✔ 3A SQL query – Apoa2, Gm2083, Alb, Apoa1, Apoe
   symbol   mean_value
0   Apoa2  7390.666667
1  Gm2083  7266.000000
2     Alb  7078.166667
3   Apoa1  6516.625000
4    Apoe  6277.500000


**3B – your answers:**
1. …
2. …
3. …
4. …

**3B – model answers**
1. **Stream** with `gzip.open(path, "rt")` and count lines (reads = lines / 4): one pass, O(1) memory – loading is impossible.
2. Load it with **pandas** using small dtypes / `usecols`, and **save it once in a binary columnar format**
   (Parquet/Feather) or SQLite, so that the daily loading is fast.
3. **Chunked aggregation** with sums and counts per (gene, condition) – or SQL `GROUP BY` in SQLite/DuckDB.
   The partial results are small (genes × conditions).
4. The median is **not decomposable** into partial sums: all values of a gene are needed. Options: sort/group on disk
   (database `ORDER BY`, DuckDB's `median`), process the file gene by gene after sorting it by gene,
   or accept an approximate (streaming) quantile.

# Wrap-up
1. **Measure first** – file size on disk ≠ memory in Python (`memory_usage(deep=True)`, `tracemalloc`).
2. **Stream** what you only need to see once: file iteration, generators, generator expressions.
3. **Load less** with pandas: `usecols`, small numeric dtypes, `category`.
4. **Chunk** large tables and combine **sums and counts**, never means of means.
5. **Store smartly** for repeated analyses: databases (SQLite) or columnar formats (Parquet).

In [ ]:
# Clean up the files created in this session (optional)
# LONG_FILE.unlink(missing_ok=True)
# DB_FILE.unlink(missing_ok=True)